# EXERCISE SOLUTION

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

In [1]:
# imports

import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama

In [2]:
# Constants

MODEL = "llama3.2"

In [3]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [4]:
# Let's try one out

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
For m

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT4o have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [5]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [6]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

## Messages

The API from Ollama expects the same message format as OpenAI:

```
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]

In [7]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

## Time to bring it together - now with Ollama instead of OpenAI

In [8]:
# And now: call the Ollama function instead of OpenAI

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']

In [9]:
summarize("https://edwarddonner.com")

'# Website Summary\n### About the Owner\nEdward Donner is a co-founder and CTO of AI startup Nebula.io, previously founder and CEO of untapt (acquired in 2021), and Managing Director at JPMorgan. He is passionate about Large Language Models (LLMs) and enjoys sharing his knowledge through Udemy courses.\n\n### Recent Posts\n#### News/Announcements\n* February 17, 2026: "Vibe Coder to Agentic Engineer – RESOURCES" (a resource post)\n* January 4, 2026: "AI Builder with n8n – Create Agents and Voice Agents – RESOURCES" (a resource post)\n* September 15, 2025: "AI Engineering MLOps Track – Deploy AI to Production – RESOURCES" (a resource post)\n* May 28, 2025: A course ordering guide for his AI courses\n\n#### Personal Message\nEdward welcomes visitors from his Udemy courses and invites them to chat with him and his Digital Avatar.'

In [10]:
# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [11]:
display_summary("https://edwarddonner.com")

# Website Summary
### Overview
This website is owned by Edward Donner, a co-founder and CTO of AI startup Nebula.io. The site appears to be a personal blog or portfolio showcasing his work in the field of Artificial Intelligence (AI).

### Content
- **Courses**: Offers Udemy courses on LLMs (Large Language Models) with over 600,000 enrolled users across 194 countries.
- **Blog Posts**:
  - "Vibe Coder to Agentic Engineer – RESOURCES" (February 17, 2026): Discusses resources for transitioning from Vibe Coder to Agentic Engineer.
  - "AI Builder with n8n – Create Agents and Voice Agents – RESOURCES" (January 4, 2026): Shares resources on building AI with n8n.
  - "AI Engineering MLOps Track – Deploy AI to Production – RESOURCES" (September 15, 2025): Discusses the process of deploying AI in production.
  - "Which order to take the AI courses?" (May 28, 2025): Offers guidance on the recommended course sequence.

### Navigation
The website includes standard navigation elements such as:
- Avatar
- Curriculum
- Proficiency
- C4
- Outsmart
- About
- Posts
- Get in touch
- Social media links.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [12]:
display_summary("https://cnn.com")

**Summary of CNN Website**

### News and Announcements

* US sets 'settlement' that could end war with Iran, but details remain unclear.
* Trump nominates Jay Clayton to top intelligence post amid controversy over prior pick.
* Pope Leo meets superstar musician Bad Bunny, but no photos were released.
* New York Knicks pull off greatest comeback in NBA Finals history.

### Global News

* Mexico defeats South Africa 2-0 in the World Cup's opening match.
* Iran's foreign ministry spokesperson says no agreement has been finalized with the US over war concerns.
* European Central Bank becomes first major bank to hike rates amid Iran war tensions.
* El Niño is strengthening rapidly, bringing severe weather to parts of Africa and Asia.

### Business

* Inflation hits a three-year high, but experts say this may not be the full story.
* Data center locations are becoming increasingly popular in the US, with thousands of Americans giving up their citizenship due to the tax benefits.

### Science and Discovery

* AI is sparking a jobs boom, despite concerns over organized crime involvement.
* Flesh-eating screwworm has reached the US for the first time since being eradicated.

### Entertainment

* Gwyneth Paltrow faces backlash for starring in luxury Israel real estate ad.
* Former child actor accuses Sean "Diddy" Combs of sexual assault in new lawsuit.

### Sports

* Knicks pull off greatest comeback in NBA Finals history, with fans celebrating the historic win.
* Severe storms wreak havoc across the Midwest.

In [13]:
display_summary("https://anthropic.com")

# Anthropic Website Summary

## Overview
Anthropic is a public benefit corporation dedicated to securing the benefits and mitigating the risks of AI. They aim to build AI to serve humanity's long-term well-being.

## Announcements

*   **Introducing The Anthropic Institute**: A new effort to confront significant challenges that powerful AI will pose to our societies.
*   **Anthropic’s Responsible Scaling Policy**: Aligning science and practice to ensure safe AI development.
*   **Claude Opus 4.8**: An upgrade to Opus, handling long-running work with consistency.
*   **Fable 5**: The next generation of intelligence, announced.

## Products

*   Claude: A suite of tools for building, deploying, and managing AI models.
*   Claude Code for Enterprise: A commercial version of the code editor.
*   Claude Cowork: A platform for remote collaboration.

## Solutions
Anthropic offers solutions in various domains, including:

*   AI agents
*   Code modernization
*   Education
*   Financial services
*   Government
*   Healthcare
*   Legal
*   Life sciences
*   Nonprofits
*   Security
*   Small business

## Pricing
Anthropic offers various pricing plans, including the Max plan, Team plan, and Enterprise plan.

## Company
The company is registered as Anthropic PBC (Public Benefit Corporation) in 2024.

## News
Please note that this information might not be up to date.

# Sharing your code

I'd love it if you share your code afterwards so I can share it with others! You'll notice that some students have already made changes (including a Selenium implementation) which you will find in the community-contributions folder. If you'd like add your changes to that folder, submit a Pull Request with your new versions in that folder and I'll merge your changes.

If you're not an expert with git (and I am not!) then GPT has given some nice instructions on how to submit a Pull Request. It's a bit of an involved process, but once you've done it once it's pretty clear. As a pro-tip: it's best if you clear the outputs of your Jupyter notebooks (Edit >> Clean outputs of all cells, and then Save) for clean notebooks.

PR instructions courtesy of an AI friend: https://chatgpt.com/share/670145d5-e8a8-8012-8f93-39ee4e248b4c